# **SIMULATOR TRADING RECORDS**
## *Primera parte: Creación de datos*


## Preanálisis (registrado antes de ejecutar la versión 2 del simulador)

*Nota de transparencia: la versión 1 del simulador ya se había ejecutado; sus fallas (sin reinversión, generador aleatorio compartido, un solo costo agregado) motivaron la versión 2. Este párrafo fija las expectativas para la versión 2 antes de correr sus estimadores; no pretende ser un registro ciego de la primera ronda.*

Inyectaremos, para cada cuenta i, δᵢ = δs·uᵢ y κᵢ = κs·vᵢ, con uᵢ, vᵢ ~ U(0,1) independientes y δs, κs ∈ {0, 0.3, 0.8} según el escenario. La probabilidad diaria de vender un lote es 0.015·(1+1.5κᵢ)·(1±δᵢ), con + si está en ganancia y − si está en pérdida. Los escenarios 7 y 8 no usan δ ni κ: el 7 vende, cada 20 días, los lotes cuyo peso supera en más de 25% el peso objetivo 1/n; el 8 vende con probabilidad 1.6 veces la base si la acción subió en los últimos 20 días y 0.6 veces si bajó. La cartera neta paga comisión de 3 pb por lado y medio spread de 2 pb por lado. Esperamos que: (i) PGR−PLR sea ≈ 0 en los escenarios 1, 4 y 5 (el IC 95% por bootstrap de cuentas incluye el cero) y crezca de forma aproximadamente proporcional a δs en los escenarios 2, 3 y 6; (ii) el turnover crezca con κs y no dependa de δs en primer orden; (iii) como los precios son exógenos y las recompras aleatorias, ninguna regla altere el rendimiento bruto esperado, por lo que la pendiente del turnover sobre el rendimiento bruto debería ser cero y la del neto negativa y del tamaño del costo por unidad de turnover; anticipamos, sin embargo, que la definición original de turnover (ventas a precios corrientes entre capital inicial) contiene un componente mecánico que producirá una pendiente positiva, y que una versión normalizada por el valor medio de la cartera será neutral en el nulo; (iv) los escenarios 7 y 8 darán PGR−PLR > 0 aun con δ = 0, de modo que ese indicador no los separa de la disposición, pero una regresión de la probabilidad de venta sobre ganancia acumulada, rendimiento reciente y sobrepeso debería asignar cada patrón a su propio regresor.

### **Inversionistas**

In [ ]:
import numpy as np
import pandas as pd

# Generar números aleatorios, el 42 ayuda a producir exactamente la misma secuencia de números "aleatorios" cada vez que se ejecute
rng = np.random.default_rng(42)

# Definimos el tamaño de la población de inversores
N = 1000

# Construimos el DataFrame 'inversionistas' que almacena las características de cada uno
inversionistas = pd.DataFrame({
    # {Odean (1998) / Account Identifier}
    # Identificador único numérico para cada inversor
    "id": np.arange(1, N + 1), #np.arange solo hace una lista de números de 1 a 1000

    # 'delta' representa la intensidad del Efecto Disposición (mu) inyectado, generado uniformemente en
    "delta": rng.uniform(0, 1, N), # rng.uniform genera número de forma uniforme de 0 a 1

    # 'kappa' representa la intensidad de la Sobreconfianza (gamma) inyectada, generada uniformemente en [1]
    "kappa": rng.uniform(0, 1, N),

    # Asignación aleatoria uniforme del capital monetario inicial entre $10,000 y $500,000 USD
    "capital_inicial": rng.uniform(10_000, 500_000, N).round(2),

    # Cantidad discreta de acciones diferentes (entre 5 y 30) que el inversor mantendrá en su portafolio
    "num_posiciones": rng.integers(5, 31, N)#rng.integers generador de números enteros de 1 a 30
})

print(inversionistas.head())
print(inversionistas.describe())
inversionistas.to_csv("inversionistas.csv", index=False)

### **Matrices de precios de acciones**

In [ ]:
# {P01.pdf / Configuración de la Simulación}
# Definimos las dimensiones del mercado: 50 acciones y 500 días de simulación
n_acciones = 50
n_dias = 500

# {Odean (1998) / Punto de Referencia Inicial}
# Fijamos el precio inicial de $100.0 USD para todas las acciones en el Día 0 para estandarizar la escala
precio_inicial = 100.0

# Generamos los rendimientos diarios aleatorios (cambios porcentuales) para cada acción y cada día.
# 'loc=0.0005' es el crecimiento diario promedio (0.05%) y 'scale=0.015' es la fluctuación diaria (1.5%)
retornos_diarios = rng.normal(loc=0.0005, scale=0.015, size=(n_dias, n_acciones))

# Creamos una matriz vacía de 501 filas (Día 0 al 500) y 50 columnas (una por cada acción)
precios = np.zeros((n_dias + 1, n_acciones)) #np.zeros crea una matriz básica llena de 0 sólo para que no esté vacía

# Asignamos el precio inicial de $100.0 a la primera fila (Día 0) para todas las acciones
precios[0, :] = precio_inicial

# RANDOM WALK para calcular el precio de cada día
# Fórmula: Precio de Hoy = Precio de Ayer * (1 + Crecimiento/Caída del Día)
for dia in range(1, n_dias + 1): #ciclo para que haga lo mismo desde el día 1 - 500
    precios[dia, :] = precios[dia - 1, :] * (1 + retornos_diarios[dia - 1, :]) #precios[dia - 1, :] busca el precio del día anterior
                                            # * (1 + retornos_diarios[dia - 1, :] se multiplica el p de ayer por el facotr crecimiento o caida de hoy
                                            # ej: si la acción sube 2% la formula sería precio de ayer * 1.02, ése se´ria el precio nuevo


# Convertimos la matriz a tabla de pandas y le asignamos los nombres
columnas = [f"Stock_{i+1}" for i in range(n_acciones)]
df_precios = pd.DataFrame(precios, columns=columnas)

# Verificamos dimensiones y resultados
print("Dimensiones de la matriz de precios:", df_precios.shape)
df_precios.to_csv("precios_mercado.csv", index_label="Dia")

In [ ]:
df_precios.head()

In [ ]:
import matplotlib.pyplot as plt

df_precios.plot(legend=False, figsize=(10, 5), alpha=0.5, title="Simulación de Precios (50 Acciones)")
plt.xlabel("Días")
plt.ylabel("Precio ($)")
plt.grid(True, alpha=0.3)
plt.show()

La matriz de precios creada se generó mediante un proceso de paseo aleatorio estocástico (Random Walk) sencillo. Este genera el precio de 50 acciones para 500 días exactamente.

Para el día 0 se fijó un precio inicial de 100 USD ($P_0 = 100$) para todos los activos, ya que esto facilita la identificación de ganancias/pérdidas sin un sesgo nominal de escala (mostrado en el paper: "The Journal of Finance - 2002, Odean - Are Investors Reluctant to Realize Their Losses").  Además, los parámetros usados para definir la Dist.normal de los precios fueron de:

* Retorno medio de 0.05% lo que equivale a un rendimiento anualizado aproximado del 12.6% $$(0.05\% \times 252)$$

* Volatilidad diaria del 1.5% equivalente a una volatilidad anualizada de 23.8% $$(1.5\% \times \sqrt{252})$$

Los cuales equivalen a el comportamiento histórico "normal" de las acciones. Por otra parte, la gráfica de los precios nos indica que en $t_0$ existe un precio de referencia inicial estandarizado, también una dispersión temporal en forma de abanico que muestra el incremento de activos con alto rendimiento. Los precios se generan antes de las decisiones y ninguna regla de venta consulta rendimientos futuros; la gráfica solo describe la dispersión y no prueba independencia.


### **Portafolio inicial por inversionista ($S_0$)**

In [ ]:
# Función para construir el port.inicial
def crear_portafolio_inicial(df_inv, df_p): #crea dos tablas df_inv será inversionistas df_p es los precios
    # Extraemos la lista con los nombres de las 50 acciones disponibles
    lista_acciones = list(df_p.columns)

    # Lista vacía donde iremos almacenando cada compra realizada en el Día 0
    posiciones_iniciales = []

    # Recorremos la tabla de los 1,000 inversores fila por fila
    for _, inv in df_inv.iterrows():
        # Extraemos el id, capital inicial y cantidad de acciones a comprar de inversionistas
        id_inv = int(inv["id"])
        capital = inv["capital_inicial"]
        n_pos = int(inv["num_posiciones"])

        # {Barber & Odean (2000) / Selección Aleatoria}
        acciones_elegidas = rng.choice(lista_acciones, size=n_pos, replace=False) #rng.choice selecciona al azar acciones para crear en cuales si invertirá cada inversionista

        # Divide el capital en partes iguales entre la cantidad de acciones elegidas
        capital_por_accion = capital / n_pos

        # Registro de la compra de cada acción
        for accion in acciones_elegidas:
            # {Odean (1998) / Punto de Referencia Base}
            # Obtenemos el precio de compra en el Día 0=100
            precio_compra = df_p.loc[0, accion]

            # Calculamos la cantidad de acciones compradas (Dinero asignado / 100)
            titulos = capital_por_accion / precio_compra

            # {Odean (1998) / Registro de Contabilidad Mental}
            # Guardamos la posición individual que servirá como precio de compra de referencia
            #aqui creamos las partes de nuestra tabla para el portafolio
            posiciones_iniciales.append({
                "id_inversor": id_inv,         # Número de cuenta del inversor (1 al 1000)
                "accion": accion,              # Nombre de la acción adquirida (ej. 'Stock_12')
                "titulos": titulos,            # Cantidad de títulos comprados
                "precio_compra": precio_compra,# Precio inicial de compra ($100 USD)
                "dia_compra": 0                # Día de la transacción (Día 0)
            })

    # Se transforma la lista en tabla
    return pd.DataFrame(posiciones_iniciales)
# Especificamos los datos que debe usar el portafolio
df_portafolio_dia0 = crear_portafolio_inicial(inversionistas, df_precios)

# Verificamos resultados
df_portafolio_dia0.to_csv("portafolio_inicial_dia0.csv", index=False)
print(df_portafolio_dia0.describe())

In [ ]:
df_portafolio_dia0.head()

El portafolio inicial que creamos distribuye el capital que le asignamos anteriormente a todos los inversionistas de forma equitativa entre los activos que tienen cada uno varía entre 5 y 30 acciones, todo asignado por medio de un muestreo aleatorio uniforme. La simulación creó $16, 833$ de posiciones de compra por tanto existe un promedio de 16.8 acciones por inversionista aproximadamente. El número de titulos que compran los inversionista varía mucho dada las posibilidades y movimientos que fueron aleatorios para cada inversionista, depende de su capital y como quiso repartirlo entre las acciones.

## *Second Part: Decision rules*

Ejecución de los escenarios

¿Qué estamos simulando?

Creamos un mercado artificial con 1,000 inversionistas, 50 acciones y 500 días. Cada inversionista comienza con diferente capital y número de posiciones. Después simulamos sus decisiones de compra y venta para estudiar dos sesgos: efecto disposición y sobreconfianza.

El parámetro δ (delta) controla el efecto disposición.
Cuando aumenta, el inversionista tiene mayor tendencia a vender posiciones ganadoras y conservar perdedoras. El parámetro κ (kappa) controla la intensidad de trading: cuando aumenta, el inversionista opera con mayor frecuencia.

Probamos ocho escenarios para separar ambos efectos. También incluimos costos de transacción y dos posibles explicaciones alternativas —rebalanceo y reversión a la media— para comprobar que un patrón aparentemente conductual puede tener más de una explicación.

Finalmente utilizamos regresiones y bootstrap para analizar la relación entre turnover, sesgos y rendimientos, y repetimos todo en 40 mercados nuevos para medir la incertidumbre entre mercados.

Diccionario:

PGR: proporción de ganancias realizadas.
PLR: proporción de pérdidas realizadas.
PGR−PLR: indicador del efecto disposición; valores positivos indican que se realizan proporcionalmente más ganancias que pérdidas.
Turnover (a): ventas a precios corrientes / capital inicial (la definición original).
Turnover normalizado (b): ventas / valor medio de la cartera.
Operaciones por posición (c): número de ventas / número de posiciones.
Riesgo: desviación estándar de los rendimientos diarios de la cartera.
Rendimiento bruto: rendimiento antes de costos.
Rendimiento neto: rendimiento después de costos.

### Diseño de la versión 2 (qué cambió y por qué)

- **Aleatoriedad local.** Cada simulación recibe una semilla explícita (`SEED_SIM`); no hay generador global y el resultado no depende del orden de ejecución. Dentro de un mismo mundo los ocho escenarios usan los mismos números aleatorios (uniformes de venta y sorteos de recompra), de modo que sus diferencias se deben al mecanismo y no al ruido de Monte Carlo.
- **Mundo.** Población, precios y portafolio son los de la semilla 42 (los de las celdas de datos). Para medir la incertidumbre entre mercados se repite todo en 40 mundos nuevos, con población, precios y portafolio regenerados (celda E).
- **Costos.** Comisión de 0.03% por lado y spread de 0.04% (0.02% por lado: se vende al bid y se compra al ask), aplicados a cada venta y a cada recompra de la cartera neta; comisión y spread se acumulan por separado. La ida y vuelta cuesta ≈ 10 pb, igual que el costo agregado de la versión 1.
- **Reglas.** `base`: probabilidad de venta 0.015·(1+1.5κᵢ)·(1±δᵢ). `rebalanceo`: base neutral más, cada 20 días, venta de los lotes con peso superior en 25% al objetivo 1/n; el lote vendido se reemplaza por uno de tamaño objetivo y el excedente se reparte entre las demás posiciones a costo promedio; no consulta el precio de compra. `reversion`: probabilidad 1.6× la base si la acción subió en los últimos 20 días y 0.6× si bajó; no consulta el precio de compra.
- **Turnover.** Se reportan tres definiciones: (a) ventas a precios corrientes / capital inicial (la original), (b) ventas / valor medio de la cartera, (c) operaciones por posición.
- **Riesgo.** Se calcula la volatilidad diaria realizada de la cartera bruta de cada cuenta y se usa como control en las regresiones.

### Módulo del simulador

La celda siguiente guarda el simulador como el archivo `p01_sim.py` (en Colab: panel *Archivos*). Sube ese mismo archivo a tu repositorio de GitHub.

In [ ]:
%%writefile p01_sim.py
"""
P01 - Finanzas conductuales: simulador v2 y estimadores.

Diseño:
  * Todo el azar sale de generadores LOCALES con semilla explícita (nada global).
  * Números aleatorios comunes (CRN): dentro de un mismo "mundo" todos los escenarios
    usan las mismas uniformes de venta y los mismos sorteos de recompra, así que las
    diferencias entre escenarios se deben al mecanismo y no al ruido de Monte Carlo.
  * Costos separados: comisión (proporcional al monto operado, por lado) y spread
    (medio spread pagado en cada lado: se vende al bid y se compra al ask).
  * Reglas de venta: "base" (δ, κ), "rebalanceo" (por desviación de peso, NO por costo
    de compra) y "reversion" (por rendimiento reciente, NO por costo de compra).
"""
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------- mundo
def generar_mundo(seed=42, N=1000, n_acciones=50, n_dias=500):
    """Población + precios + portafolio inicial. generar_mundo(42) reproduce
    exactamente inversionistas / df_precios / df_portafolio_dia0 del notebook."""
    rng = np.random.default_rng(seed)
    inv = pd.DataFrame({
        "id": np.arange(1, N + 1),
        "delta": rng.uniform(0, 1, N),
        "kappa": rng.uniform(0, 1, N),
        "capital_inicial": rng.uniform(10_000, 500_000, N).round(2),
        "num_posiciones": rng.integers(5, 31, N),
    })
    ret = rng.normal(0.0005, 0.015, size=(n_dias, n_acciones))
    precios = np.zeros((n_dias + 1, n_acciones))
    precios[0, :] = 100.0
    for d in range(1, n_dias + 1):
        precios[d, :] = precios[d - 1, :] * (1 + ret[d - 1, :])
    cols = [f"Stock_{i+1}" for i in range(n_acciones)]
    cuenta, accion = [], []
    for _, fila in inv.iterrows():
        elegidas = rng.choice(cols, size=int(fila["num_posiciones"]), replace=False)
        for a in elegidas:
            cuenta.append(int(fila["id"]) - 1)          # índice 0..N-1
            accion.append(int(a.split("_")[1]) - 1)     # índice de columna 0..49
    cuenta = np.array(cuenta)
    accion = np.array(accion)
    titulos = (inv["capital_inicial"].to_numpy() / inv["num_posiciones"].to_numpy())[cuenta] / 100.0
    return {"inv": inv, "precios": precios, "cuenta": cuenta, "accion": accion,
            "titulos": titulos, "seed": seed, "N": N, "n_dias": n_dias,
            "n_acciones": n_acciones}


# ------------------------------------------------------------------ operar (1 cartera)
def _operar(tit, inv, ph, pnew, v_azar, v_reb, nl, N, c, half):
    """Ejecuta ventas y recompras sobre el vector de títulos `tit` (una cartera).
    c = comisión por lado; half = medio spread por lado (0 y 0 para la cartera bruta).
    Devuelve (tit_nuevo, comisiones_por_cuenta, spreads_por_cuenta, top_idx, top_sh)."""
    val = tit * ph
    Vi = np.bincount(inv, weights=val, minlength=N)
    target = Vi / nl
    nueva = tit.copy()
    com = np.zeros(N)
    spr = np.zeros(N)

    # -- ventas por azar: se vende el lote completo y se reinvierte todo en otra acción
    ia = np.flatnonzero(v_azar)
    if ia.size:
        v = val[ia]
        s_spr = v * half
        s_com = (v - s_spr) * c
        cash = v - s_spr - s_com
        b_com = cash * c
        cash2 = cash - b_com
        sh = cash2 / (pnew[ia] * (1 + half))
        b_spr = cash2 - sh * pnew[ia]
        nueva[ia] = sh
        com += np.bincount(inv[ia], weights=s_com + b_com, minlength=N)
        spr += np.bincount(inv[ia], weights=s_spr + b_spr, minlength=N)

    # -- rebalanceo: vende el lote sobreponderado, compra un lote nuevo de tamaño objetivo
    #    y reparte el excedente entre los lotes que no se vendieron (costo promedio).
    top_idx = np.array([], dtype=int)
    top_sh = np.array([])
    ir = np.flatnonzero(v_reb)
    if ir.size:
        v = val[ir]
        s_spr = v * half
        s_com = (v - s_spr) * c
        cash = v - s_spr - s_com
        S = np.bincount(inv[ir], weights=cash, minlength=N)
        k = np.bincount(inv[ir], minlength=N)
        nosold = ~(v_azar | v_reb)
        m = np.bincount(inv[nosold], minlength=N)
        cf = np.minimum(target, S / np.maximum(k, 1))
        E = S - k * cf
        sin_m = (m == 0) & (k > 0)
        cf = np.where(sin_m, S / np.maximum(k, 1), cf)
        E = np.where(sin_m, 0.0, E)
        # lotes nuevos
        cfi = cf[inv[ir]]
        b_com = cfi * c
        cash2 = cfi - b_com
        sh = cash2 / (pnew[ir] * (1 + half))
        b_spr = cash2 - sh * pnew[ir]
        nueva[ir] = sh
        com += np.bincount(inv[ir], weights=s_com + b_com, minlength=N)
        spr += np.bincount(inv[ir], weights=s_spr + b_spr, minlength=N)
        # top-ups
        nt = np.flatnonzero(nosold & (k[inv] > 0) & (m[inv] > 0))
        if nt.size:
            ct = E[inv[nt]] / m[inv[nt]]
            b_com = ct * c
            cash2 = ct - b_com
            sh_add = cash2 / (ph[nt] * (1 + half))
            b_spr = cash2 - sh_add * ph[nt]
            nueva[nt] = tit[nt] + sh_add
            com += np.bincount(inv[nt], weights=b_com, minlength=N)
            spr += np.bincount(inv[nt], weights=b_spr, minlength=N)
            top_idx, top_sh = nt, sh_add
    return nueva, com, spr, top_idx, top_sh


# ------------------------------------------------------------------------ simulador
def simular(mundo, delta_s, kappa_s, regla="base", seed=0,
            comision=0.0003, spread=0.0004, p_base=0.015,
            L=20, K_reb=20, banda=0.25, guardar_panel=False, n_panel=200):
    """Corre un escenario. Devuelve dict con: df (una fila por cuenta) y panel (opcional).
    regla: 'base' | 'rebalanceo' | 'reversion'."""
    inv_df = mundo["inv"]
    N = mundo["N"]
    n_dias = mundo["n_dias"]
    n_acc = mundo["n_acciones"]
    precios = mundo["precios"]
    inv = mundo["cuenta"].copy()
    stk = mundo["accion"].copy()
    tb = mundo["titulos"].copy()           # cartera bruta
    tn = mundo["titulos"].copy()           # cartera neta (paga costos)
    pc = np.full(len(inv), 100.0)          # base de costo (precio de compra del lote)
    nl = np.bincount(inv, minlength=N).astype(float)
    n_lotes = len(inv)

    delta_i = inv_df["delta"].to_numpy() * delta_s
    kappa_i = inv_df["kappa"].to_numpy() * kappa_s
    freq_lote = (1.0 + 1.5 * kappa_i)[inv]
    delta_lote = delta_i[inv]

    rng_u = np.random.default_rng([seed, 1])   # uniformes de venta (CRN)
    rng_c = np.random.default_rng([seed, 2])   # acción de recompra (CRN)
    half = spread / 2.0

    RG = np.zeros(N); PG = np.zeros(N); RL = np.zeros(N); PL = np.zeros(N)
    ntr = np.zeros(N); vsales = np.zeros(N); vacum = np.zeros(N)
    com_tot = np.zeros(N); spr_tot = np.zeros(N)
    r1 = np.zeros(N); r2 = np.zeros(N)                 # para la volatilidad realizada
    Vprev = np.bincount(inv, weights=tb * precios[0][stk], minlength=N)
    panel = []

    for dia in range(1, n_dias + 1):
        P = precios[dia]
        ph = P[stk]
        gan = ph > pc
        perd = ph < pc
        ref = precios[max(dia - L, 0)]
        r20 = (P / ref - 1.0)[stk]

        if regla == "base":
            p = np.where(gan, p_base * (1 + delta_lote),
                         np.where(perd, p_base * (1 - delta_lote), 0.0)) * freq_lote
        elif regla == "reversion":          # cree que lo que subió recientemente va a bajar
            p = np.where(r20 > 0, p_base * 1.6, p_base * 0.6) * freq_lote
        elif regla == "rebalanceo":         # base neutral; el sesgo entra por el peso
            p = np.full(n_lotes, p_base) * freq_lote
        else:
            raise ValueError(regla)
        p = np.clip(p, 0.0, 1.0)

        u = rng_u.random(n_lotes)
        c_new = rng_c.integers(0, n_acc, n_lotes)
        v_azar = u < p

        val_b = tb * ph
        Vi = np.bincount(inv, weights=val_b, minlength=N)
        w_rel = val_b / (Vi[inv] / nl[inv])          # peso relativo al objetivo 1/n
        v_reb = np.zeros(n_lotes, dtype=bool)
        if regla == "rebalanceo" and dia % K_reb == 0:
            v_reb = (w_rel > 1.0 + banda) & ~v_azar
        vende = v_azar | v_reb

        vacum += Vi
        rd = Vi / Vprev - 1.0                          # rendimiento diario de mercado de la cartera
        r1 += rd; r2 += rd * rd; Vprev = Vi
        if guardar_panel and dia % 5 == 0:
            sel = (inv < n_panel) & (gan | perd)
            panel.append(pd.DataFrame({
                "cuenta": inv[sel], "dia": dia, "gan": gan[sel].astype(int),
                "r20_pos": (r20[sel] > 0).astype(int),
                "sobrepeso": (w_rel[sel] > 1.0 + banda).astype(int),
                "vende": vende[sel].astype(int)}))

        # ---- conteos de Odean (solo en días en que la cuenta vende algo)
        if vende.any():
            hay = np.bincount(inv[vende], minlength=N) > 0
            mant = hay[inv] & ~vende
            RG += np.bincount(inv[vende & gan], minlength=N)
            RL += np.bincount(inv[vende & perd], minlength=N)
            PG += np.bincount(inv[mant & gan], minlength=N)
            PL += np.bincount(inv[mant & perd], minlength=N)
            ntr += np.bincount(inv[vende], minlength=N)
            vsales += np.bincount(inv[vende], weights=val_b[vende], minlength=N)

            pnew = P[c_new]
            tb_n, _, _, t_idx, t_sh = _operar(tb, inv, ph, pnew, v_azar, v_reb, nl, N, 0.0, 0.0)
            tn_n, cm, sp, _, _ = _operar(tn, inv, ph, pnew, v_azar, v_reb, nl, N, comision, half)
            com_tot += cm; spr_tot += sp
            # base de costo: lotes nuevos = precio de hoy; top-ups = costo promedio
            if t_idx.size:
                pc[t_idx] = (tb[t_idx] * pc[t_idx] + t_sh * ph[t_idx]) / (tb[t_idx] + t_sh)
            pc[vende] = pnew[vende]
            stk[vende] = c_new[vende]
            tb, tn = tb_n, tn_n

    Pf = precios[n_dias][stk]
    vf_b = np.bincount(inv, weights=tb * Pf, minlength=N)
    vf_n = np.bincount(inv, weights=tn * Pf, minlength=N)
    cap = inv_df["capital_inicial"].to_numpy()
    npos = inv_df["num_posiciones"].to_numpy()
    df = pd.DataFrame({
        "id": inv_df["id"], "delta": inv_df["delta"], "kappa": inv_df["kappa"],
        "capital_inicial": cap, "num_posiciones": npos,
        "delta_i": delta_i, "kappa_i": kappa_i,
        "RG": RG, "PG": PG, "RL": RL, "PL": PL,
        "num_trades": ntr, "valor_negociado": vsales,
        "turnover": vsales / cap,                                  # definición original
        "turnover_norm": vsales / (vacum / n_dias),                # ventas / valor medio de la cartera
        "ops_pos": ntr / npos,                                     # operaciones por posición
        "comisiones": com_tot, "spreads": spr_tot, "costos_totales": com_tot + spr_tot,
        "riesgo": np.sqrt(np.maximum(r2 / n_dias - (r1 / n_dias) ** 2, 0.0)),   # vol. diaria realizada
        "rend_bruto": (vf_b - cap) / cap, "rend_neto": (vf_n - cap) / cap,
    })
    out = {"df": df}
    if guardar_panel:
        out["panel"] = pd.concat(panel, ignore_index=True)
    return out


# ------------------------------------------------------------------------ estimadores
def indicadores(df):
    RG, PG, RL, PL = (df[c].sum() for c in ["RG", "PG", "RL", "PL"])
    pgr = RG / (RG + PG); plr = RL / (RL + PL)
    return {"PGR": pgr, "PLR": plr, "D": pgr - plr, "Q": pgr / plr}


def bootstrap_cuentas(df, B=1000, seed=0):
    """EE de D y Q remuestreando CUENTAS completas (conserva los 4 conteos juntos)."""
    rng = np.random.default_rng(seed)
    M = df[["RG", "PG", "RL", "PL"]].to_numpy()
    idx = rng.integers(0, len(df), size=(B, len(df)))
    S = M[idx].sum(axis=1)
    pgr = S[:, 0] / (S[:, 0] + S[:, 1]); plr = S[:, 2] / (S[:, 2] + S[:, 3])
    return {"EE_D": (pgr - plr).std(ddof=1), "EE_Q": (pgr / plr).std(ddof=1)}


def regresion(df, y, turnover_col, con_turnover=True):
    import statsmodels.api as sm
    d = df.copy()
    d["capital_miles"] = d["capital_inicial"] / 1000
    cols = ([turnover_col] if con_turnover else [])
    cols += [c for c in ["delta_i", "kappa_i"] if d[c].std() > 0]
    cols += ["capital_miles", "num_posiciones", "riesgo"]
    X = sm.add_constant(d[cols])
    return sm.OLS(d[y], X).fit(cov_type="HC1")


def lpm_panel(panel):
    """Prob. diaria de vender ~ ganancia + rendimiento reciente + sobrepeso (EE agrupados por cuenta)."""
    import statsmodels.api as sm
    X = sm.add_constant(panel[["gan", "r20_pos", "sobrepeso"]])
    return sm.OLS(panel["vende"], X).fit(cov_type="cluster",
                                          cov_kwds={"groups": panel["cuenta"]})


ESCENARIOS = {
    "1. Nulo":               dict(delta_s=0.0, kappa_s=0.0, regla="base"),
    "2. Disposición baja":   dict(delta_s=0.3, kappa_s=0.0, regla="base"),
    "3. Disposición alta":   dict(delta_s=0.8, kappa_s=0.0, regla="base"),
    "4. Frecuencia baja":    dict(delta_s=0.0, kappa_s=0.3, regla="base"),
    "5. Frecuencia alta":    dict(delta_s=0.0, kappa_s=0.8, regla="base"),
    "6. Ambos activos":      dict(delta_s=0.8, kappa_s=0.8, regla="base"),
    "7. Rebalanceo":         dict(delta_s=0.0, kappa_s=0.0, regla="rebalanceo"),
    "8. Reversión":          dict(delta_s=0.0, kappa_s=0.0, regla="reversion"),
}


# ------------------------------------------------------------ utilidades de alto nivel
def correr_todos(mundo, seed, guardar_panel=True, **kw):
    """Corre los 8 escenarios sobre un mismo mundo (números aleatorios comunes)."""
    return {n: simular(mundo, seed=seed, guardar_panel=guardar_panel, **p, **kw)
            for n, p in ESCENARIOS.items()}


def tabla_resumen(res, B=1000, seed=99):
    filas = []
    for n, r in res.items():
        df = r["df"]
        ind = indicadores(df)
        bs = bootstrap_cuentas(df, B=B, seed=seed)
        filas.append({
            "Escenario": n, "PGR": ind["PGR"], "PLR": ind["PLR"], "D": ind["D"],
            "EE_D": bs["EE_D"], "Q": ind["Q"], "EE_Q": bs["EE_Q"],
            "Turnover": df["turnover"].mean(), "Turnover_norm": df["turnover_norm"].mean(),
            "Ops_por_pos": df["ops_pos"].mean(),
            "Bruto": df["rend_bruto"].mean(), "Neto": df["rend_neto"].mean(),
            "Brecha_pp": (df["rend_bruto"] - df["rend_neto"]).mean() * 100,
            "Comisiones_USD": df["comisiones"].mean(), "Spreads_USD": df["spreads"].mean()})
    return pd.DataFrame(filas)


def tabla_regresiones(res, medidas=("turnover", "turnover_norm", "ops_pos")):
    filas = []
    for n, r in res.items():
        for tc in medidas:
            rb = regresion(r["df"], "rend_bruto", tc)
            rn = regresion(r["df"], "rend_neto", tc)
            filas.append({"Escenario": n, "Medida": tc,
                          "b_bruto": rb.params[tc], "EE_bruto": rb.bse[tc],
                          "b_neto": rn.params[tc], "EE_neto": rn.bse[tc],
                          "Dif_neto_bruto": rn.params[tc] - rb.params[tc], "R2_bruto": rb.rsquared})
    return pd.DataFrame(filas)


def mc_mundos(n_mundos=40, seed0=1000, B=300):
    """Repite todo el experimento en n_mundos mercados y poblaciones NUEVOS.
    Es la fuente correcta de incertidumbre entre mercados (~4 s por mundo)."""
    filas = []
    for w in range(n_mundos):
        seed = seed0 + w
        mundo = generar_mundo(seed)
        for nombre, par in ESCENARIOS.items():
            df = simular(mundo, seed=seed, **par)["df"]
            ind = indicadores(df)
            bs = bootstrap_cuentas(df, B=B, seed=seed)
            f = {"mundo": seed, "Escenario": nombre, "D": ind["D"], "EE_D": bs["EE_D"],
                 "Turnover": df["turnover"].mean(), "Turnover_norm": df["turnover_norm"].mean(),
                 "Bruto": df["rend_bruto"].mean(), "Neto": df["rend_neto"].mean(),
                 "EE_bruto_cuentas": df["rend_bruto"].std(ddof=1) / np.sqrt(len(df))}
            for tc in ("turnover", "turnover_norm"):
                f["b_bruto_" + tc] = regresion(df, "rend_bruto", tc).params[tc]
                f["b_neto_" + tc] = regresion(df, "rend_neto", tc).params[tc]
            filas.append(f)
    return pd.DataFrame(filas)


In [ ]:
# ============ CELDA A · Configuración y mundo ============
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import p01_sim as S                      # p01_sim.py va en la misma carpeta que el notebook

SEED_MUNDO = 42          # población + precios + portafolio inicial (igual que las celdas de datos)
SEED_SIM   = 2026        # ventas y recompras (independiente de la semilla anterior)

mundo = S.generar_mundo(SEED_MUNDO)

# Comprobación: es exactamente el mundo generado por las celdas de arriba
assert mundo["inv"].equals(inversionistas)
assert np.array_equal(mundo["precios"], df_precios.to_numpy())
assert len(mundo["cuenta"]) == len(df_portafolio_dia0)
print("Mundo idéntico al de las celdas de datos ✔")

In [ ]:
# ============ CELDA B · Los 8 escenarios (números aleatorios comunes) ============
res = S.correr_todos(mundo, seed=SEED_SIM)        # ≈ 8 s
tabla = S.tabla_resumen(res, B=1000)              # PGR, PLR, D, Q con EE por bootstrap de cuentas

# Compatibilidad con tus gráficas anteriores (celdas de figuras 2 y 3)
df_resultados_finales = pd.DataFrame({
    "Escenario": tabla["Escenario"],
    "delta": [p["delta_s"] for p in S.ESCENARIOS.values()],
    "kappa": [p["kappa_s"] for p in S.ESCENARIOS.values()],
    "PGR-PLR": tabla["D"],
    "Turnover_Promedio": tabla["Turnover"],
    "Rendimiento_Bruto": tabla["Bruto"],
    "Rendimiento_Neto": tabla["Neto"]})

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(tabla.round(5).to_string(index=False))
tabla.to_csv("resultados_escenarios.csv", index=False)

### Resultados: disposición y actividad

En el nulo D = 0.0002 (EE 0.0003; IC 95% de −0.0004 a 0.0009): compatible con cero. D crece con δs: 0.0161 con δs=0.3 y 0.0440 con δs=0.8, y Q pasa de 1.00 a 1.33 y 2.29. Con κ solo (escenarios 4 y 5) D queda en 0.0000 (EE 0.0003) mientras el turnover normalizado sube de 7.52 a 9.24 y 12.12. Con ambos mecanismos D = 0.0489, ligeramente mayor que con disposición sola (0.0440): existe una pequeña interacción (+0.005, presente en los 40 mercados de la celda E) cuyo origen no se investigó.

El rendimiento bruto medio del mercado 42 va de 42.4% a 44.0% entre escenarios, pero en 40 mercados nuevos es 28.5%–28.6% en todos: las diferencias dentro de un mercado son ruido de decisión y no un efecto de los mecanismos. El rendimiento neto es menor por 1.07 pp en el nulo y por 1.72 pp con κs=0.8; la brecha crece con la actividad porque cada operación paga comisión y spread. Estos niveles de rendimiento no estiman el rendimiento del mercado: la trayectoria de la semilla 42 dio +42.4% cuando el esperado con esos parámetros es +28.4% (ver celda E).

In [ ]:
# ============ CELDA C · ¿Qué mide realmente la pendiente del turnover? ============
reg = S.tabla_regresiones(res)          # 3 definiciones de turnover × 8 escenarios, EE robustos HC1
print(reg.round(5).to_string(index=False))

print("\nCorrelación con el rendimiento bruto:")
for n, r in res.items():
    d = r["df"]
    print(f"{n:20s} turnover={d.turnover.corr(d.rend_bruto):6.3f} | "
          f"turnover_norm={d.turnover_norm.corr(d.rend_bruto):6.3f} | "
          f"ops_pos={d.ops_pos.corr(d.rend_bruto):6.3f}")

# Mediación (escenario 6): la regresión con y sin turnover
d6  = res["6. Ambos activos"]["df"]
con = S.regresion(d6, "rend_bruto", "turnover")
sin = S.regresion(d6, "rend_bruto", "turnover", con_turnover=False)
esp = S.regresion(d6, "turnover", "turnover", con_turnover=False)      # turnover ~ κ_i
print("\nCON turnover :", con.params[["turnover", "delta_i", "kappa_i"]].round(4).to_dict())
print("SIN turnover :", sin.params[["delta_i", "kappa_i"]].round(4).to_dict(),
      "| t =", sin.tvalues[["delta_i", "kappa_i"]].round(2).to_dict())
print(f"d turnover / d κ_i = {esp.params['kappa_i']:.2f}  →  β(κ_i) espejo esperado = "
      f"{-con.params['turnover'] * esp.params['kappa_i']:.3f}  (observado {con.params['kappa_i']:.3f})")

# Independencia de los parámetros y relación con la conducta resultante
print("\ncorr(sorteo δ, sorteo κ) =", round(mundo["inv"][["delta", "kappa"]].corr().iloc[0, 1], 5))
for n in ["2. Disposición baja", "3. Disposición alta", "6. Ambos activos"]:
    d = res[n]["df"]
    print(f"{n:20s} corr(δ_i, turnover)={d.delta_i.corr(d.turnover):7.4f} | "
          f"corr(δ_i, turnover_norm)={d.delta_i.corr(d.turnover_norm):7.4f} | "
          f"corr(δ_i, κ_i)={d.delta_i.corr(d.kappa_i) if d.kappa_i.std() > 0 else float('nan'):7.5f}")

### Por qué la pendiente del turnover sale positiva

Con la definición original (ventas a precios corrientes entre capital inicial) la pendiente sobre el rendimiento bruto es 0.048 (EE 0.0045) **en el escenario nulo**, donde todas las cuentas siguen exactamente la misma regla, y es positiva en los 40 mercados de la celda E (media 0.049). No puede reflejar sobreconfianza ni habilidad: es un vínculo mecánico, porque el numerador suma ventas valoradas a precios corrientes y el denominador es fijo, de modo que una cartera que se aprecia más vende montos mayores. La prueba es que, al normalizar por el valor medio de la cartera o al contar operaciones por posición, la pendiente en el nulo es −0.003 (EE 0.006) y la correlación con el rendimiento pasa de 0.39 a −0.02.

El coeficiente de κᵢ (−0.584 con turnover en la regresión) es el reflejo de lo mismo: −β(turnover) × (∂turnover/∂κᵢ) = −0.046 × 12.4 ≈ −0.57. Sin turnover en la regresión, ni κᵢ ni δᵢ afectan el rendimiento (t = −0.97 y 0.89). Las asociaciones «significativas» de la versión anterior venían de controlar por un mediador. El efecto de los costos sí se recupera: la diferencia neto−bruto de la pendiente es −0.0013 en el nulo (−0.0013 a −0.0017 en los ocho escenarios), del tamaño de los 10 pb de ida y vuelta capitalizados; en 40 mercados su desviación estándar es 0.00008, aunque cada pendiente por separado es ruidosa (EE ≈ 0.006).

Con disposición activa la pendiente normalizada deja de ser cero (0.012 con δs=0.3; 0.034 con δs=0.8; 0.028 en el escenario 6). Ya no es un artefacto de medición sino causalidad inversa: las cuentas que ganan más venden más lotes ganadores. La pendiente positiva del turnover puede tener, por tanto, dos orígenes distintos que un solo coeficiente no separa.

In [ ]:
# ============ CELDA D · Validación (nulo, monotonicidad, estimadores cruzados) ============
T = tabla.set_index("Escenario")

# (1) Recuperación del nulo
n1 = T.loc["1. Nulo"]
print(f"NULO: D = {n1.D:.5f}, EE = {n1.EE_D:.5f}, IC95% = [{n1.D-1.96*n1.EE_D:.5f}, {n1.D+1.96*n1.EE_D:.5f}]")

# (2) Estimadores cruzados: ¿cada estimador reacciona SOLO a su mecanismo?
rn = reg[reg.Medida == "turnover_norm"].set_index("Escenario")
cruzada = pd.DataFrame({"D": T.D, "EE_D": T.EE_D, "Turnover_norm": T.Turnover_norm,
                        "β bruto turnover_norm": rn.b_bruto, "EE β": rn.EE_bruto})
print("\n", cruzada.round(5).to_string())

# (3) Monotonicidad con malla fina, promediando 10 mundos nuevos (≈ 1 min)
niveles, filas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], []
for w in range(10):
    m = S.generar_mundo(2000 + w)
    for v in niveles:
        a = S.simular(m, v, 0.0, "base", seed=2000 + w)["df"]     # solo δ
        b = S.simular(m, 0.0, v, "base", seed=2000 + w)["df"]     # solo κ
        filas.append({"nivel": v, "D | solo δ": S.indicadores(a)["D"],
                      "D | solo κ": S.indicadores(b)["D"],
                      "turnover_norm | solo κ": b.turnover_norm.mean(),
                      "turnover_norm | solo δ": a.turnover_norm.mean()})
malla = pd.DataFrame(filas).groupby("nivel").mean()
print("\n", malla.round(5).to_string())

### Validación

**Recuperación del nulo.** D = 0.0002 con EE 0.0003. En 40 mercados nuevos, D del nulo tiene media 0.0001 y desviación 0.0003; el z = D/EE tiene desviación 0.93 y rechaza |z|>1.96 en 2.5% de los mercados (esperado ≈ 5%). Con κ solo rechaza en 5.0% (ambos escenarios).

**Monotonicidad.** Con malla fina (10 mercados, δs de 0 a 1) D = 0.0001, 0.0108, 0.0216, 0.0325, 0.0439, 0.0558: estrictamente creciente y casi lineal (≈ 0.055·δs). Con κs de 0 a 1 el turnover normalizado sube de 7.50 a 13.11 y D queda en cero. En 40 de 40 mercados D(1) < D(2) < D(3) y turnover(1) < turnover(4) < turnover(5).

**Estimadores cruzados.** El indicador de disposición permanece en cero cuando solo se mueve κ. La pendiente del turnover normalizado también es cero con κ solo (−0.005 y −0.003; EE 0.005), pero no lo es con δ (0.012 y 0.034): la disposición contamina ese estimador por causalidad inversa (ver celda C). La validación cruzada se cumple en un sentido y falla en el otro. Con δ activo, el turnover normalizado incluso baja para δs > 0.4 (7.59 con δs=0.4 a 6.97 con δs=1.0), coherente con que retener perdedoras reduce las ventas.

In [ ]:
# ============ CELDA E · Incertidumbre entre mercados (Monte Carlo de mundos) ============
# Cada mundo regenera población, precios y portafolio. ≈ 3 minutos con 40 mundos.
mc = S.mc_mundos(n_mundos=40)
mc.to_csv("mc_mundos.csv", index=False)
g = mc.groupby("Escenario")
resumen_mc = pd.DataFrame({
    "D media": g.D.mean(), "SD(D) entre mundos": g.D.std(), "EE(D) por cuentas": g.EE_D.mean(),
    "Bruto medio": g.Bruto.mean(), "SD bruto entre mundos": g.Bruto.std(),
    "EE bruto por cuentas": g.EE_bruto_cuentas.mean(),
    "β turnover": g.b_bruto_turnover.mean(), "β turnover_norm": g.b_bruto_turnover_norm.mean(),
    "SD β_norm": g.b_bruto_turnover_norm.std()})
print(resumen_mc.round(5).to_string())

nulo = mc[mc.Escenario == "1. Nulo"]
z = nulo.D / nulo.EE_D
print(f"\nNULO en {len(nulo)} mundos: media z = {z.mean():.2f}, SD z = {z.std():.2f}, "
      f"rechazo |z|>1.96 = {(z.abs() > 1.96).mean():.1%} (esperado ≈ 5%)")
print("β turnover (definición original) > 0 en", int((nulo.b_bruto_turnover > 0).sum()), "de", len(nulo), "mundos")

### Errores estándar, clustering e incertidumbre entre mercados

El error estándar de D se obtiene remuestreando cuentas completas (1,000 réplicas, conservando juntos los cuatro conteos de cada cuenta). Se contrasta con la desviación de D entre 40 mercados con población, precios y portafolio regenerados. En los escenarios base coinciden (nulo: 0.00032 por cuentas frente a 0.00029 entre mercados; disposición alta: 0.00109 frente a 0.00115). No coinciden en los confusores, cuyo D depende de la trayectoria de precios: rebalanceo 0.00032 frente a 0.00084 (2.6 veces) y reversión 0.00036 frente a 0.00057 (1.6 veces). Para esos escenarios el bootstrap de cuentas subestima la incertidumbre.

Con el rendimiento la diferencia es mucho mayor: el error estándar por cuentas es 0.37 pp, pero entre mercados la desviación estándar es 5.6 pp. El mercado de la semilla 42 dio 42.4% de rendimiento bruto medio, unas 2.4 desviaciones sobre el promedio de los 40 mercados (28.6%, cercano al 28.4% teórico). Por eso los niveles de rendimiento de un solo mercado no se interpretan como estimaciones del mercado, y las comparaciones entre escenarios sólo son válidas condicionalmente al mundo, gracias a los números aleatorios comunes.

Las regresiones usan una observación por cuenta con errores robustos HC1. El panel lote-día de la celda F usa errores agrupados por cuenta, porque un mismo lote aparece en muchos días. No se agrupa por acción: cada cuenta mantiene varias acciones y no hay anidamiento; la dependencia entre cuentas proviene de la trayectoria común de precios y se cubre repitiendo el mercado.

In [ ]:
# ============ CELDA F · ¿Qué datos separan disposición, rebalanceo y reversión? ============
# Panel lote-día (200 cuentas, un día de cada 5). Modelo lineal de probabilidad con EE agrupados por cuenta.
partes = []
for n in ["1. Nulo", "3. Disposición alta", "7. Rebalanceo", "8. Reversión"]:
    m_ = S.lpm_panel(res[n]["panel"])
    partes.append(pd.DataFrame({"Escenario": n,
                                "coef (pp de prob. diaria de vender)": m_.params * 100,
                                "t": m_.tvalues}).drop("const"))
print(pd.concat(partes).round(3).to_string())
print("\nD (PGR−PLR) de esos escenarios:", tabla.set_index("Escenario").loc[
      ["1. Nulo", "3. Disposición alta", "7. Rebalanceo", "8. Reversión"], "D"].round(4).to_dict())

### Escenarios 7 y 8: confusores con mecanismo propio

En la versión 1 los confusores eran multiplicadores sobre la misma variable que usa la disposición (ganancia frente al precio de compra), por lo que D > 0 era inevitable y no informaba nada. En la versión 2 cada uno decide con una variable distinta y no consulta el precio de compra (rebalanceo: peso; reversión: rendimiento de 20 días). Ambos producen D > 0 con δ = 0: 0.0098 (EE 0.0003) y 0.0230 (EE 0.0004), frente a 0.0440 con disposición alta. El indicador capta la correlación entre la ganancia sobre el costo y esas otras variables, de modo que **no identifica la motivación**: una D positiva es compatible con disposición, rebalanceo o reversión, y la disposición baja (0.0161) produce más señal que el rebalanceo.

Con las variables correctas, un panel lote-día (200 cuentas, un día de cada cinco; 336,400 observaciones; modelo lineal de probabilidad con errores agrupados por cuenta) sí los separa. Disposición: la ganancia acumulada eleva la probabilidad diaria de vender en 1.20 pp (t = 17.8) y el rendimiento reciente y el sobrepeso no importan (≈ 0). Reversión: el rendimiento positivo de 20 días eleva la probabilidad en 1.40 pp (t = 30.5) y los demás regresores no importan. Rebalanceo: el sobrepeso la eleva en 32.9 pp (t = 86.8) y los demás no importan. En el nulo los tres son cero.

Este resultado depende de observar sin error las variables que generan las decisiones. Con datos reales no se observan pesos objetivo, pronósticos ni expectativas, y ahí el indicador PGR−PLR por sí solo no basta: por eso la tabla de datos necesarios de la sección 8 del reporte. Limitaciones: la reversión es una regla sobre el rendimiento pasado en un mercado sin reversión real, y el rebalanceo es por bandas cada 20 días; ninguno modela expectativas explícitas.

In [ ]:
# ============ CELDA G · Gráfica de D con intervalos ============
etq = []
for n, p in S.ESCENARIOS.items():
    sub = f"δs={p['delta_s']}, κs={p['kappa_s']}" if p["regla"] == "base" else f"regla: {p['regla']}"
    etq.append(f"{n}\n{sub}")
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.bar(range(8), tabla["D"], yerr=1.96 * tabla["EE_D"], capsize=4)
ax.axhline(0, lw=1)
ax.set_xticks(range(8)); ax.set_xticklabels(etq, rotation=35, ha="right")
ax.set_ylabel("PGR − PLR  (IC 95% por bootstrap de cuentas)")
ax.set_title("Efecto disposición por escenario")
plt.tight_layout(); plt.show()

Las barras muestran D con IC 95% por bootstrap de cuentas. Los escenarios 1, 4 y 5 son compatibles con cero; 2, 3 y 6 lo superan de forma monotónica; 7 y 8 también lo superan sin ninguna disposición inyectada.

In [ ]:
# 2. TURNOVER

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    df_resultados_finales["Escenario"],
    df_resultados_finales["Turnover_Promedio"]
)

plt.title(
    "Turnover promedio por escenario"
)

plt.ylabel(
    "Turnover promedio"
)

plt.xlabel(
    "Escenario"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

El turnover (ventas a precios corrientes / capital inicial) pasa de 8.79 en el nulo a 10.80 con κs=0.3 y a 14.17 con κs=0.8; con disposición sola apenas cambia (9.08 y 9.10). Como muestra la celda C, esta definición mezcla actividad con rendimiento; la versión normalizada sube de 7.52 a 9.24 y 12.12.

In [ ]:
# 3. RENDIMIENTO BRUTO VS NETO

x = np.arange(
    len(df_resultados_finales)
)

ancho = 0.35

plt.figure(
    figsize=(11, 5)
)

plt.bar(
    x - ancho / 2,
    df_resultados_finales[
        "Rendimiento_Bruto"
    ],
    width=ancho,
    label="Bruto"
)

plt.bar(
    x + ancho / 2,
    df_resultados_finales[
        "Rendimiento_Neto"
    ],
    width=ancho,
    label="Neto"
)

plt.title(
    "Rendimiento bruto y neto por escenario"
)

plt.ylabel(
    "Rendimiento promedio"
)

plt.xlabel(
    "Escenario"
)

plt.xticks(
    x,
    df_resultados_finales["Escenario"],
    rotation=45,
    ha="right"
)

plt.legend()

plt.tight_layout()

plt.show()

El rendimiento bruto medio va de 42.4% a 44.0% entre escenarios en este mercado, pero es 28.5%–28.6% en todos al promediar 40 mercados: la diferencia es ruido. El neto es menor en todos: 1.07 pp en el nulo, 1.31 pp con κs=0.3 y 1.72 pp con κs=0.8. La brecha crece con la actividad porque cada operación paga comisión y spread.

In [ ]:
# ============ CELDA H · Reproducibilidad ============
import statsmodels
res2 = S.correr_todos(S.generar_mundo(SEED_MUNDO), seed=SEED_SIM, guardar_panel=False)
tabla2 = S.tabla_resumen(res2, B=1000)
assert tabla.equals(tabla2), "Los resultados NO se reproducen"
print("Segunda corrida idéntica a la primera ✔")
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| statsmodels", statsmodels.__version__)

### Reproducibilidad

Los resultados dependen únicamente de `SEED_MUNDO=42`, `SEED_SIM=2026` y del módulo `p01_sim.py`; no hay estado global ni dependencia del orden de las celdas. La celda anterior repite la corrida completa y verifica igualdad exacta. Las semillas de los 40 mercados adicionales son 1000 a 1039 y las de la malla de monotonicidad 2000 a 2009. Para reproducir: *Entorno → Reiniciar y ejecutar todo* con `p01_sim.py` en la misma carpeta (probado con Python 3.12 y statsmodels 0.15 en dos combinaciones, numpy 2.4 + pandas 3.0 y numpy 2.0 + pandas 2.2, con resultados idénticos). Repositorio: <pon aquí tu URL de GitHub con `p01_sim.py`, el notebook y `requirements.txt`>.

### Conclusiones

1. Con el mecanismo δ inyectado, PGR−PLR responde de forma monotónica y casi lineal (≈ 0.055·δs), el nulo es compatible con cero y la sobreconfianza (κ) no genera una falsa disposición. Su error estándar por cuentas es fiable en los escenarios base (desviación entre 40 mercados 0.0003, igual al EE del nulo).
2. El mismo indicador es positivo con rebalanceo (0.0098) y con reversión (0.0230) sin ninguna disposición: una D positiva no identifica la motivación. Con datos de peso y rendimiento reciente, un panel lote-día sí separa los tres mecanismos.
3. La pendiente negativa esperada del turnover sobre el rendimiento neto no aparece con la definición original: en el nulo la pendiente es +0.048 y positiva en 40 de 40 mercados, por cómo se mide el turnover. Normalizado, es cero (−0.003, EE 0.006) y el efecto de costos (≈ −0.0013 por unidad) se recupera con precisión. Con disposición, la pendiente normalizada es positiva (0.012 a 0.034) por causalidad inversa: quien gana más vende más.

### Limitaciones

- Las tablas principales provienen de un solo mercado, cuyo rendimiento (+42.4%) es unas 2.4 desviaciones superior al promedio de 40 mercados; los niveles de rendimiento no son estimaciones del mercado, y las comparaciones entre escenarios son condicionales al mundo.
- Los valores de comisión y spread son supuestos.
- Rebalanceo y reversión son reglas reducidas (bandas de peso cada 20 días; rendimiento de 20 días), sin expectativas ni pesos objetivo heterogéneos; el mercado no tiene reversión real.
- La sobreconfianza se modela solo como mayor frecuencia de operación.
- Existe una pequeña interacción δ×κ (+0.005 en D) cuyo origen no se investigó.
- El panel lote-día usa 200 cuentas y un día de cada cinco.